In [3]:
# Stop old session - Force kill
import os
import signal
import subprocess

# Method 1: Stop active session
from pyspark.sql import SparkSession
try:
    spark = SparkSession.getActiveSession()
    if spark is not None:
        # Stop the SparkContext first
        spark.sparkContext.stop()
        spark.stop()
        print('✅ Old session stopped via SparkSession.stop()')
except Exception as e:
    print(f'⚠️ Error stopping session: {e}')

# Method 2: Kill Java processes
try:
    subprocess.run(['pkill', '-9', '-f', 'java'], stderr=subprocess.DEVNULL)
    print('✅ Killed old Java processes')
except:
    pass

import time
time.sleep(2)
print('✅ Ready for new session')

✅ Old session stopped via SparkSession.stop()
✅ Killed old Java processes


ERROR:root:Exception while sending command.
Traceback (most recent call last):
  File "/opt/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 516, in send_command
    raise Py4JNetworkError("Answer from Java side is empty")
py4j.protocol.Py4JNetworkError: Answer from Java side is empty

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/opt/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/opt/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 539, in send_command
    raise Py4JNetworkError(
py4j.protocol.Py4JNetworkError: Error while sending or receiving


✅ Ready for new session


In [ ]:
# Create new session - fresh start
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName('LAB5-CAU2') \
    .master('spark://spark-master:7077') \
    .config('spark.hadoop.fs.defaultFS', 'hdfs://hadoop-master:9000') \
    .getOrCreate()

print('✅ New Spark Session created!')
print(f'Spark Version: {spark.version}')
print(f'Master: {spark.sparkContext.master}')

ERROR:root:Exception while sending command.
Traceback (most recent call last):
  File "/opt/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 516, in send_command
    raise Py4JNetworkError("Answer from Java side is empty")
py4j.protocol.Py4JNetworkError: Answer from Java side is empty

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/opt/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/opt/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 539, in send_command
    raise Py4JNetworkError(
py4j.protocol.Py4JNetworkError: Error while sending or receiving


Py4JError: SparkConf does not exist in the JVM

In [2]:
df = spark.read.csv('hdfs://hadoop-master:9000/shop_transactions.csv', header=True, inferSchema=True)
print(f'✅ Data loaded from HDFS: {df.count():,} rows')
df.printSchema()
df.show(5)

✅ Data loaded from HDFS: 1,000,000 rows
root
 |-- price: double (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- customerRegion: string (nullable = true)
 |-- productType: string (nullable = true)

+------+--------+--------------+-----------+
| price|quantity|customerRegion|productType|
+------+--------+--------------+-----------+
| 78.04|       3|        Europe|       Home|
|190.39|       5|  NorthAmerica|Electronics|
|147.74|       3|  SouthAmerica|       Home|
|121.74|       4|  NorthAmerica|     Beauty|
| 35.42|       6|          Asia|       Home|
+------+--------+--------------+-----------+
only showing top 5 rows



In [3]:
from pyspark.sql.functions import col

df = df.withColumn('revenue', col('price') * col('quantity'))
df.createOrReplaceTempView('shop')
print('✅ Temp view created: shop')

✅ Temp view created: shop


In [4]:
result = spark.sql('''
SELECT 
    productType,
    COUNT(*) as count,
    ROUND(AVG(price), 2) as avg_price,
    ROUND(AVG(quantity), 2) as avg_quantity,
    ROUND(SUM(price * quantity), 2) as total_revenue,
    ROUND(AVG(price * quantity), 2) as avg_revenue
FROM shop
GROUP BY productType
ORDER BY total_revenue DESC
''')

print('📊 Hive Analysis Results:')
result.show()

📊 Hive Analysis Results:
+-----------+------+---------+------------+--------------+-----------+
|productType| count|avg_price|avg_quantity| total_revenue|avg_revenue|
+-----------+------+---------+------------+--------------+-----------+
|     Beauty|200164|   102.52|         5.0| 1.027066273E8|     513.11|
|   Clothing|200308|   102.44|         5.0|1.0266763331E8|     512.55|
|Electronics|199599|    102.6|        5.01|1.0263895848E8|     514.23|
|       Home|199744|    102.7|         5.0|1.0253354155E8|     513.32|
|     Sports|200185|   102.57|        4.99|1.0248298796E8|     511.94|
+-----------+------+---------+------------+--------------+-----------+



In [5]:
top5 = spark.sql('''
SELECT 
    productType,
    ROUND(SUM(price * quantity), 2) as total_revenue,
    COUNT(*) as transaction_count
FROM shop
GROUP BY productType
ORDER BY total_revenue DESC
LIMIT 5
''')

print('🏆 Top 5 Products by Revenue:')
top5.show()

🏆 Top 5 Products by Revenue:
+-----------+--------------+-----------------+
|productType| total_revenue|transaction_count|
+-----------+--------------+-----------------+
|     Beauty| 1.027066273E8|           200164|
|   Clothing|1.0266763331E8|           200308|
|Electronics|1.0263895848E8|           199599|
|       Home|1.0253354155E8|           199744|
|     Sports|1.0248298796E8|           200185|
+-----------+--------------+-----------------+



In [6]:
import time

# Job 1: Revenue by Region
print('CÂU 1: JOB PERFORMANCE')
print('=' * 60)

start = time.time()
job1 = spark.sql('SELECT customerRegion, ROUND(SUM(price*quantity), 2) as revenue FROM shop GROUP BY customerRegion ORDER BY revenue DESC')
job1.collect()
time1 = time.time() - start
print(f'Job 1 Time: {time1:.3f}s')
job1.show()

# Job 2: Stats by Product
start = time.time()
job2 = spark.sql('SELECT productType, COUNT(*) as cnt, ROUND(AVG(price), 2) as avg_price, ROUND(SUM(price*quantity), 2) as revenue FROM shop GROUP BY productType ORDER BY revenue DESC')
job2.collect()
time2 = time.time() - start
print(f'Job 2 Time: {time2:.3f}s')
job2.show()

print(f'\nPerformance: Job1={time1:.3f}s, Job2={time2:.3f}s')

CÂU 1: JOB PERFORMANCE
Job 1 Time: 1.201s
+--------------+--------------+
|customerRegion|       revenue|
+--------------+--------------+
|  NorthAmerica|1.2847607974E8|
|        Europe|1.2835354112E8|
|          Asia|1.2810554747E8|
|  SouthAmerica|1.2809458027E8|
+--------------+--------------+

Job 2 Time: 1.222s
+-----------+------+---------+--------------+
|productType|   cnt|avg_price|       revenue|
+-----------+------+---------+--------------+
|     Beauty|200164|   102.52| 1.027066273E8|
|   Clothing|200308|   102.44|1.0266763331E8|
|Electronics|199599|    102.6|1.0263895848E8|
|       Home|199744|    102.7|1.0253354155E8|
|     Sports|200185|   102.57|1.0248298796E8|
+-----------+------+---------+--------------+


Performance: Job1=1.201s, Job2=1.222s


In [7]:
performance_data = spark.createDataFrame([
    ('Job1_Time_Seconds', str(round(time1, 3))),
    ('Job2_Time_Seconds', str(round(time2, 3)))
], ['metric', 'value'])

performance_data.write.mode('overwrite').csv('hdfs://hadoop-master:9000/lab5_cau1_performance', header=True)
print('✅ CÂU 1 saved to /lab5_cau1_performance')

✅ CÂU 1 saved to /lab5_cau1_performance


In [8]:
from pyspark.sql.functions import col, sum as spark_sum, count, round as spark_round

print('CÂU 2: HIVE ANALYSIS - TOP 5 PRODUCTTYPE')
print('=' * 60)

# Hive Query: Doanh thu theo productType
hive_result = spark.sql('''
SELECT 
    productType,
    COUNT(*) as transaction_count,
    ROUND(AVG(price), 2) as avg_price,
    ROUND(AVG(quantity), 2) as avg_quantity,
    ROUND(SUM(price * quantity), 2) as total_revenue
FROM shop
GROUP BY productType
ORDER BY total_revenue DESC
''')

hive_result.show()

# Top 5 productType
top5_result = spark.sql('''
SELECT 
    productType,
    ROUND(SUM(price * quantity), 2) as total_revenue,
    COUNT(*) as transaction_count
FROM shop
GROUP BY productType
ORDER BY total_revenue DESC
LIMIT 5
''')

print('\nTop 5 ProductType by Revenue:')
top5_result.show()

# Lưu vào HDFS
top5_result.write.mode('overwrite').csv('hdfs://hadoop-master:9000/q2_hive_top5_productType', header=True)
print('✅ Saved to HDFS: /q2_hive_top5_productType')

CÂU 2: HIVE ANALYSIS - TOP 5 PRODUCTTYPE
+-----------+-----------------+---------+------------+--------------+
|productType|transaction_count|avg_price|avg_quantity| total_revenue|
+-----------+-----------------+---------+------------+--------------+
|     Beauty|           200164|   102.52|         5.0| 1.027066273E8|
|   Clothing|           200308|   102.44|         5.0|1.0266763331E8|
|Electronics|           199599|    102.6|        5.01|1.0263895848E8|
|       Home|           199744|    102.7|         5.0|1.0253354155E8|
|     Sports|           200185|   102.57|        4.99|1.0248298796E8|
+-----------+-----------------+---------+------------+--------------+


Top 5 ProductType by Revenue:
+-----------+--------------+-----------------+
|productType| total_revenue|transaction_count|
+-----------+--------------+-----------------+
|     Beauty| 1.027066273E8|           200164|
|   Clothing|1.0266763331E8|           200308|
|Electronics|1.0263895848E8|           199599|
|       Home|1.

In [9]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, StandardScaler, VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

print('\nCÂU 3: LOGISTIC REGRESSION')
print('=' * 60)

# Chuẩn bị dữ liệu
df_model = spark.sql('SELECT price, quantity, customerRegion, productType FROM shop')

# Indexing
region_indexer = StringIndexer(inputCol='customerRegion', outputCol='customerRegion_idx')
product_indexer = StringIndexer(inputCol='productType', outputCol='productType_idx')

# Vector assembler (trước khi standardize)
vector_assembler = VectorAssembler(inputCols=['price', 'quantity', 'customerRegion_idx'], outputCol='features_raw')

# Standardize (Z-score)
scaler = StandardScaler(inputCol='features_raw', outputCol='features', withMean=True, withStd=True)

# Model
lr = LogisticRegression(labelCol='productType_idx', featuresCol='features', maxIter=10)

# Pipeline
pipeline = Pipeline(stages=[region_indexer, product_indexer, vector_assembler, scaler, lr])

# Train/Test split (80/20)
train_data, test_data = df_model.randomSplit([0.8, 0.2], seed=42)
print(f'Training set: {train_data.count():,} rows')
print(f'Test set: {test_data.count():,} rows')

# Train model
model = pipeline.fit(train_data)
print('✅ Model trained')

# Evaluate
predictions = model.transform(test_data)

# Metrics
accuracy_evaluator = MulticlassClassificationEvaluator(labelCol='productType_idx', predictionCol='prediction', metricName='accuracy')
accuracy = accuracy_evaluator.evaluate(predictions)

precision_evaluator = MulticlassClassificationEvaluator(labelCol='productType_idx', predictionCol='prediction', metricName='weightedPrecision')
precision = precision_evaluator.evaluate(predictions)

recall_evaluator = MulticlassClassificationEvaluator(labelCol='productType_idx', predictionCol='prediction', metricName='weightedRecall')
recall = recall_evaluator.evaluate(predictions)

f1_evaluator = MulticlassClassificationEvaluator(labelCol='productType_idx', predictionCol='prediction', metricName='f1')
f1 = f1_evaluator.evaluate(predictions)

print(f'\n📊 Model Metrics:')
print(f'Accuracy:  {accuracy:.4f}')
print(f'Precision: {precision:.4f}')
print(f'Recall:    {recall:.4f}')
print(f'F1-Score:  {f1:.4f}')

# Lưu metrics
metrics_data = spark.createDataFrame([
    ('Accuracy', str(round(accuracy, 4))),
    ('Precision', str(round(precision, 4))),
    ('Recall', str(round(recall, 4))),
    ('F1_Score', str(round(f1, 4)))
], ['metric', 'value'])

metrics_data.write.mode('overwrite').csv('hdfs://hadoop-master:9000/q3_logistic_regression_metrics', header=True)
print('✅ Metrics saved to HDFS: /q3_logistic_regression_metrics')

# Lưu predictions
predictions.select('price', 'quantity', 'customerRegion', 'productType', 'prediction').write.mode('overwrite').csv('hdfs://hadoop-master:9000/q3_logistic_regression_predictions', header=True)
print('✅ Predictions saved to HDFS: /q3_logistic_regression_predictions')


CÂU 3: LOGISTIC REGRESSION
Training set: 799,886 rows
Test set: 200,114 rows
✅ Model trained

📊 Model Metrics:
Accuracy:  0.1995
Precision: 0.1992
Recall:    0.1995
F1-Score:  0.1546
✅ Metrics saved to HDFS: /q3_logistic_regression_metrics
✅ Predictions saved to HDFS: /q3_logistic_regression_predictions
